# Quenya-TTS: training run on Kaggle

Fine-tunes the pretrained multilingual ToucanTTS model on Quenya recordings, on Kaggle's free GPU.

Before running:
1. **Settings → Accelerator → GPU T4 x2** (one GPU is used), and **Settings → Internet → on** (needs a phone-verified Kaggle account).
2. **Add Input → your dataset** with the recordings and their texts, laid out as `mp3/<name>.mp3` and `texts/<name>.txt` (one line of Quenya per line, in the order it is read). The first cell finds it under `/kaggle/input`.

Recordings whose licence does not allow publishing a model trained on them (such as Glǽmscrafu, for now) are for experiments only: keep the dataset private and do not publish the checkpoint.

In [ ]:
import glob, os
# Kaggle mounts datasets at different depths (/kaggle/input/<name> or /kaggle/input/datasets/<owner>/<name>),
# and an upload may keep or flatten its folders, so look for the recordings and the texts anywhere under it.
def folder_with(extension):
    files = glob.glob(f"/kaggle/input/**/*{extension}", recursive=True)
    if not files:
        raise FileNotFoundError(f"no {extension} file under /kaggle/input: add the dataset with Add Input")
    return os.path.dirname(sorted(files)[0])
AUDIO, TEXTS = folder_with(".mp3"), folder_with(".txt")
print("recordings:", AUDIO, len(glob.glob(f"{AUDIO}/*.mp3")), "| texts:", TEXTS, len(glob.glob(f"{TEXTS}/*.txt")))
CORPUS = "Glaemscrafu"                          # corpus name, as quenya_tts.train expects it
STEPS = 500                                     # 500 for a first test run; a few thousand for a real one
PY = "/kaggle/working/py311/bin/python"       # Toucan needs Python 3.10/3.11; Kaggle's own Python is newer

## 1. Install

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null 2>&1 || true
apt-get -qq install -y espeak-ng libportaudio2 > /dev/null
cd /kaggle/working  # a rerun may start inside the folder deleted below
rm -rf /kaggle/working/Quenya-TTS /kaggle/working/py311  # start clean when the cell is run again
git clone -q --recurse-submodules -b claude/project-thread-dgfuie https://github.com/brunopacheco1/Quenya-TTS.git /kaggle/working/Quenya-TTS
cd /kaggle/working/Quenya-TTS
# Toucan pins packages (torch 2.4, numpy 1.23, librosa 0.9) that have no builds for Kaggle's Python, so everything
# goes into a separate Python 3.11 environment, made with uv. The GUI-only packages are left out: training does
# not need them and some do not install on a headless machine.
pip install -q uv
uv venv -q --seed --python 3.11 /kaggle/working/py311  # --seed adds pip, which phonepiece imports
grep -viE '^(pyqt5|pyqtgraph|pynput|PyAutoGUI|gradio)' requirements.txt > /tmp/requirements.txt
uv pip install -q --python /kaggle/working/py311/bin/python -r /tmp/requirements.txt "transformers>=4.46,<4.50"
/kaggle/working/py311/bin/python -c "import torch; print('torch', torch.__version__, 'GPU:', torch.cuda.is_available())"

## 2. Cut the recordings into one clip per line

Writes the clips and `transcript.txt` to `Corpora/Quenya/<CORPUS>`. Lines the phone recogniser hears too differently from the text are left out; `cuts.csv` lists every line with its phone error rate.

In [ ]:
%cd /kaggle/working/Quenya-TTS
!{PY} -m quenya_tts.prepare_corpus --audio "{AUDIO}" --texts "{TEXTS}" --name {CORPUS} --device cuda
import pandas as pd
cuts = pd.read_csv(f"Corpora/Quenya/{CORPUS}/cuts.csv")
print(cuts.kept.value_counts())
cuts.sort_values("per", ascending=False).head(15)

## 3. Train

In [ ]:
!{PY} -m quenya_tts.train --gpu_id 0 --corpora {CORPUS} --steps {STEPS} --batch_size 8

## 4. Listen

In [ ]:
import glob
from IPython.display import Audio, display
model = sorted(glob.glob("Models/ToucanTTS_Quenya*/best.pt"))[-1]
tests = {
    "elen_sila": "Elen síla lúmenn' omentielvo.",
    "ilye": "Ar ilyë tier undulávë lumbulë.",
    "valimar": "Namárië! Nai hiruvalyë Valimar.",
    "dialogue": "Ma quetelyë Quenya? Quetin Quenya.",
}
for name, text in tests.items():
    !{PY} -m quenya_tts.read "{text}" --model {model} --device cuda -o /kaggle/working/{name}.wav
    print(text)
    display(Audio(f"/kaggle/working/{name}.wav"))

## 5. Keep the results

Download `best.pt` from the model folder and the WAVs from `/kaggle/working` (Output tab, or the file browser on the right). Toucan's progress plots (`*.png` in the model folder) show the loss going down.